# Airport network analysis with GraphFrames

This notebook builds a directed graph of US airports and runs the graph algorithms that GraphFrames exposes in Python. PageRank, shortest paths, connected components, and label propagation are GraphX algorithms. GraphFrames calls them from a DataFrame API, which is the Python interface on Databricks.

Each airport is a vertex. Each directed route with at least one flight is an edge. The edge attributes are the number of flights and the average departure and arrival delay.

Run the cells from top to bottom. The Spark session stays up after the last cell.


## 1. Setup

Attach the notebook to an all-purpose cluster running **Databricks Runtime ML** with **Dedicated** access mode. Runtime ML already includes GraphFrames. Standard access and serverless compute do not work, because GraphFrames needs the SparkContext.

The next cell does three things:

1. Names the two tables the notebook reads.
2. Sets a checkpoint folder. Connected components and label propagation save intermediate results there.
3. Builds a test graph with three airports and runs connected components on it.

GraphFrames is ready when the test shows A, B, and C with the same component number. If `dbfs:/tmp` is rejected, change the checkpoint path to a folder in a volume you can write.


In [1]:
!pip install graphframes
from graphframes import GraphFrame
from pyspark.sql import functions as F

AIRPORTS = "dbacademy.default.airports"
FLIGHTS = "dbacademy.default.t_ontime_reporting"



test_graph = GraphFrame(
    spark.createDataFrame([("A",), ("B",), ("C",)], ["id"]),
    spark.createDataFrame([("A", "B"), ("B", "C")], ["src", "dst"]),
)
display(test_graph.connectedComponents())


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 4.7 MB/s eta 0:00:00

[notice] A new release of pip is available: 23.0.1 -> 26.3
[notice] To update, run: pip install --upgrade pip


NameError: name 'spark' is not defined

## 2. Load the two tables

`spark.table` reads a Unity Catalog table by its three-part name: catalog, schema, table. Both frames are cached because the later sections scan them more than once.

### Columns used from `dbacademy.default.airports`

| Column | Role in the graph |
| --- | --- |
| `iata` | Vertex id |
| `name` | Airport name |
| `city`, `state` | Labels on the tables |
| `latitude`, `longitude` | Map coordinates |

### Columns used from `dbacademy.default.t_ontime_reporting`

| Column | Role in the graph |
| --- | --- |
| `ORIGIN`, `DEST` | IATA codes. These become `src` and `dst` |
| `ORIGIN_AIRPORT_ID` | Numeric Bureau of Transportation Statistics id |
| `ORIGIN_CITY_NAME` | City label |
| `DEP_DELAY`, `ARR_DELAY` | Delay in minutes. Blank when the delay was not recorded |
| `OP_CARRIER_FL_NUM` | Flight number. This extract has no airline code |


In [ ]:
airports_df = spark.table(AIRPORTS).cache()
flights_df = spark.table(FLIGHTS).cache()

print(f"{airports_df.count():,} airports from {AIRPORTS}")
airports_df.printSchema()
display(airports_df.limit(5))

print(f"{flights_df.count():,} flights from {FLIGHTS}")
flights_df.printSchema()
display(flights_df.limit(5))


## 3. Explore routes before building the graph

A vertex is an airport that appears as an origin or a destination. An edge is a distinct `(ORIGIN, DEST)` pair. The same flight number can belong to different airlines, and this extract does not include a carrier column, so the useful counts are cities and routes.


In [ ]:
print(f"Origin airports: {flights_df.select('ORIGIN').distinct().count():,}")
print(f"Destination airports: {flights_df.select('DEST').distinct().count():,}")

print("Busiest origins")
display(
    flights_df.groupBy("ORIGIN", "ORIGIN_CITY_NAME")
    .count()
    .orderBy(F.desc("count"))
    .limit(10)
)

print("Busiest routes")
display(
    flights_df.groupBy("ORIGIN", "DEST")
    .agg(
        F.count("*").alias("flights"),
        F.round(F.avg(F.col("DEP_DELAY").cast("double")), 1).alias("avg_dep_delay"),
        F.round(F.avg(F.col("ARR_DELAY").cast("double")), 1).alias("avg_arr_delay"),
    )
    .orderBy(F.desc("flights"))
    .limit(10)
)


## 4. The same questions in Spark SQL

The DataFrame aggregations above are registered as views so the next queries can use SQL. `DEP_DELAY` is cast to double because a blank delay is null, and `AVG` skips nulls.

The state count reads the full airports file, including small fields with no airline flight in this extract.


In [ ]:
airports_df.createOrReplaceTempView("airports")
flights_df.createOrReplaceTempView("flights")

display(spark.sql("""
    SELECT
        ORIGIN AS src,
        DEST AS dst,
        COUNT(*) AS flights,
        ROUND(AVG(CAST(DEP_DELAY AS DOUBLE)), 1) AS avg_dep_delay
    FROM flights
    GROUP BY ORIGIN, DEST
    ORDER BY flights DESC
    LIMIT 10
"""))

display(spark.sql("""
    SELECT state, COUNT(*) AS airports
    FROM airports
    GROUP BY state
    ORDER BY airports DESC
    LIMIT 10
"""))


## 5. Vertices

The vertex id is the IATA code, stored as a string. The airports file also contains fields that never appear in this flight extract. Those fields stay out of the graph so degree and PageRank describe the flown network.

Flight airports missing from the airports table stay in the graph. The IATA code fills in for the missing name.


In [ ]:
endpoints = (
    flights_df.select(F.col("ORIGIN").cast("string").alias("id"))
    .union(flights_df.select(F.col("DEST").cast("string").alias("id")))
    .distinct()
)

vertices = (
    endpoints.join(airports_df, endpoints.id == F.col("iata").cast("string"), "left")
    .select(
        endpoints.id.alias("id"),
        F.coalesce(F.col("name"), endpoints.id).alias("airport_name"),
        F.col("city"),
        F.col("state"),
        F.col("latitude").cast("double").alias("latitude"),
        F.col("longitude").cast("double").alias("longitude"),
    )
    .cache()
)

print(f"Vertices: {vertices.count():,}")
display(vertices.orderBy("id").limit(5))

coords = (
    vertices.filter(F.col("latitude").isNotNull() & F.col("longitude").isNotNull())
    .select("id", "latitude", "longitude")
    .toPandas()
)
airport_coords = {
    row.id: (row.latitude, row.longitude) for row in coords.itertuples(index=False)
}
print(f"Airports with coordinates: {len(airport_coords):,}")


## 6. Edges

Each edge is one directed route. `flight_count` is the number of flights on that route. The delay columns are averages over flights that recorded a delay.

GraphFrames requires the column names `src` and `dst`.


In [ ]:
edges = (
    flights_df.select(
        F.col("ORIGIN").cast("string").alias("src"),
        F.col("DEST").cast("string").alias("dst"),
        F.col("DEP_DELAY").cast("double").alias("dep_delay"),
        F.col("ARR_DELAY").cast("double").alias("arr_delay"),
    )
    .groupBy("src", "dst")
    .agg(
        F.count("*").alias("flight_count"),
        F.round(F.avg("dep_delay"), 2).alias("avg_dep_delay"),
        F.round(F.avg("arr_delay"), 2).alias("avg_arr_delay"),
    )
    .cache()
)

print(f"Edges: {edges.count():,}")
display(edges.orderBy(F.desc("flight_count")).limit(5))

display(spark.sql("""
    SELECT
        ORIGIN AS src,
        DEST AS dst,
        COUNT(*) AS flight_count,
        ROUND(AVG(CAST(DEP_DELAY AS DOUBLE)), 2) AS avg_dep_delay
    FROM flights
    GROUP BY ORIGIN, DEST
    ORDER BY flight_count DESC
    LIMIT 5
"""))


## 7. Build the GraphFrame

`cache()` keeps the vertex and edge frames for the algorithms that follow. Degree counts incident routes. It is the sum of in-degree and out-degree, so a route flown in both directions contributes two.


In [ ]:
graph = GraphFrame(vertices, edges)
print(f"Graph vertices: {graph.vertices.count():,}")
print(f"Graph edges: {graph.edges.count():,}")


## 8. Map the busiest routes

The axes cover the continental United States. Routes in Alaska, Hawaii, and the territories are in the edge table and fall outside this window. Line width follows `flight_count`. The labels are the eight airports with the most plotted flights.


In [ ]:
import matplotlib.pyplot as plt

top_routes = edges.orderBy(F.desc("flight_count")).limit(50).toPandas()
print(f"Routes loaded for the map: {len(top_routes)}")


def plot_top_routes():
    fig, ax = plt.subplots(figsize=(15, 10))
    plotted = {}
    for edge in top_routes.itertuples(index=False):
        if edge.src not in airport_coords or edge.dst not in airport_coords:
            continue
        src_lat, src_lon = airport_coords[edge.src]
        dst_lat, dst_lon = airport_coords[edge.dst]
        width = max(0.5, min(5, edge.flight_count / 500))
        ax.plot([src_lon, dst_lon], [src_lat, dst_lat], color="steelblue", alpha=0.45, linewidth=width)
        plotted[edge.src] = (src_lat, src_lon)
        plotted[edge.dst] = (dst_lat, dst_lon)

    traffic = {}
    for airport_id, (lat, lon) in plotted.items():
        ax.scatter(lon, lat, s=40, c="firebrick", alpha=0.9, edgecolors="black", linewidths=0.4, zorder=3)
        traffic[airport_id] = top_routes.loc[
            (top_routes["src"] == airport_id) | (top_routes["dst"] == airport_id),
            "flight_count",
        ].sum()

    for airport_id, _ in sorted(traffic.items(), key=lambda item: item[1], reverse=True)[:8]:
        lat, lon = plotted[airport_id]
        ax.annotate(airport_id, (lon, lat), xytext=(4, 4), textcoords="offset points", fontsize=9, fontweight="bold")

    ax.set_title("Top 50 routes, continental window")
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    ax.set_xlim(-125, -65)
    ax.set_ylim(25, 50)
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    plt.show()


plot_top_routes()


## 9. All airports, with the main hubs marked

Gray points are every airport in `dbacademy.default.airports`. Red points are the 15 vertices with the highest degree in the flight graph. The window is again the continental United States, and the cell prints how many airports fall outside it.


In [ ]:
import matplotlib.pyplot as plt

all_airports = (
    airports_df.filter(F.col("latitude").isNotNull() & F.col("longitude").isNotNull())
    .select(
        F.col("iata").cast("string").alias("id"),
        F.col("latitude").cast("double").alias("latitude"),
        F.col("longitude").cast("double").alias("longitude"),
    )
    .toPandas()
)
hubs = (
    graph.degrees.join(vertices, "id")
    .select("id", "latitude", "longitude", "degree")
    .filter(F.col("latitude").isNotNull())
    .orderBy(F.desc("degree"))
    .limit(15)
    .toPandas()
)

outside = all_airports[
    (all_airports["longitude"] < -130)
    | (all_airports["longitude"] > -60)
    | (all_airports["latitude"] < 20)
    | (all_airports["latitude"] > 55)
]
print(f"Airports outside the continental window: {len(outside):,}")


def plot_airports():
    fig, ax = plt.subplots(figsize=(15, 10))
    ax.scatter(
        all_airports["longitude"],
        all_airports["latitude"],
        s=12,
        c="lightsteelblue",
        alpha=0.7,
        linewidths=0,
    )
    ax.scatter(
        hubs["longitude"],
        hubs["latitude"],
        s=80,
        c="firebrick",
        alpha=0.9,
        edgecolors="black",
        linewidths=0.6,
        zorder=3,
    )
    for hub in hubs.itertuples(index=False):
        ax.annotate(
            hub.id,
            (hub.longitude, hub.latitude),
            xytext=(4, 4),
            textcoords="offset points",
            fontsize=8,
            fontweight="bold",
        )
    ax.set_title("Airports in the file, top 15 hubs by degree")
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    ax.set_xlim(-130, -60)
    ax.set_ylim(20, 55)
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    plt.show()


plot_airports()


## 10. Degree

`degrees` counts routes, in each direction. `inDegrees` counts incoming routes and `outDegrees` counts outgoing routes. An airport with a nonstop to 100 other airports has degree near 200 when those routes are flown both ways.

The SQL query under the degree tables counts flights instead of routes. A hub can lead one list and not the other.


In [ ]:
in_degrees = graph.inDegrees
out_degrees = graph.outDegrees
degrees = graph.degrees.cache()
degrees.count()

print("Highest degree")
display(
    degrees.join(vertices, "id")
    .select("id", "airport_name", "city", "state", "degree")
    .orderBy(F.desc("degree"))
    .limit(10)
)

print("Highest in-degree")
display(
    in_degrees.join(vertices, "id")
    .select("id", "airport_name", "city", "state", "inDegree")
    .orderBy(F.desc("inDegree"))
    .limit(10)
)

print("Highest out-degree")
display(
    out_degrees.join(vertices, "id")
    .select("id", "airport_name", "city", "state", "outDegree")
    .orderBy(F.desc("outDegree"))
    .limit(10)
)


In [ ]:
print("Departing flights by airport")
display(spark.sql("""
    SELECT
        a.iata,
        a.name,
        a.city,
        a.state,
        COUNT(*) AS departing_flights
    FROM flights f
    JOIN airports a ON f.ORIGIN = a.iata
    GROUP BY a.iata, a.name, a.city, a.state
    ORDER BY departing_flights DESC
    LIMIT 10
"""))


## 11. Shortest paths

`shortestPaths` reports hop count. A nonstop route is 1 hop. The landmark is the vertex id, which is the IATA code. Atlanta is `ATL`. The numeric id `10397` is `ORIGIN_AIRPORT_ID` for the same airport, and it is a different column.


In [ ]:
display(
    flights_df.filter(F.col("ORIGIN") == "ATL")
    .select("ORIGIN", "ORIGIN_AIRPORT_ID", "ORIGIN_CITY_NAME")
    .distinct()
)

HUB = "ATL"
paths = (
    graph.shortestPaths(landmarks=[HUB])
    .select("id", "distances")
    .join(vertices, "id")
    .select(
        "id",
        "airport_name",
        "city",
        "state",
        F.col("distances").getItem(HUB).alias("hops_from_atl"),
    )
    .cache()
)

print("Airports farthest from ATL, in hops")
display(paths.orderBy(F.desc_nulls_last("hops_from_atl")).limit(10))


def hops_between(start_id, end_id):
    """Hop count from start_id to end_id. The landmark is the destination."""
    row = (
        graph.shortestPaths(landmarks=[end_id])
        .select("id", "distances")
        .filter(F.col("id") == start_id)
        .select(F.col("distances").getItem(end_id).alias("hops"))
        .collect()
    )
    if not row or row[0]["hops"] is None:
        return f"No path from {start_id} to {end_id}"
    return f"{start_id} to {end_id}: {int(row[0]['hops'])} hops"


print(hops_between("ATL", "SEA"))
print(hops_between("ATL", "LAX"))


## 12. Motifs

A motif is a structural pattern written as a path. The triangle query finds directed cycles of three airports: `a → b → c → a`. The same cycle is returned once for each starting airport.

The second query finds pairs with a flight in each direction. The filter keeps each unordered pair once.


In [ ]:
triangles = (
    graph.find("(a)-[ab]->(b); (b)-[bc]->(c); (c)-[ca]->(a)")
    .select(
        F.col("a.id").alias("a"),
        F.col("b.id").alias("b"),
        F.col("c.id").alias("c"),
        F.col("a.airport_name").alias("a_name"),
        F.col("b.airport_name").alias("b_name"),
        F.col("c.airport_name").alias("c_name"),
    )
    .cache()
)
print(f"Directed triangle embeddings: {triangles.count():,}")
display(triangles.limit(10))

both_ways = (
    graph.find("(a)-[]->(b); (b)-[]->(a)")
    .filter(F.col("a.id") < F.col("b.id"))
    .select(
        F.col("a.id").alias("a"),
        F.col("b.id").alias("b"),
        F.col("a.airport_name").alias("a_name"),
        F.col("b.airport_name").alias("b_name"),
    )
    .distinct()
    .cache()
)
print(f"Airport pairs with flights both ways: {both_ways.count():,}")
display(both_ways.orderBy("a", "b").limit(10))


## 13. PageRank

PageRank scores airports by the routes that point at them. Every route counts as one edge. `flight_count` remains an attribute of that edge. GraphFrames reads an edge column named `weight` when it is present. This graph leaves that column out, so PageRank follows the route structure.

The comparison table puts route degree, departing flights, and PageRank on the same rows.


In [ ]:
pagerank_results = graph.pageRank(resetProbability=0.15, maxIter=10)
ranks = pagerank_results.vertices.select("id", "pagerank").cache()
ranks.count()

print("Highest PageRank")
display(
    ranks.join(vertices, "id")
    .select("id", "airport_name", "city", "state", F.round("pagerank", 4).alias("pagerank"))
    .orderBy(F.desc("pagerank"))
    .limit(10)
)

departures = edges.groupBy(F.col("src").alias("id")).agg(
    F.sum("flight_count").alias("departing_flights")
)
comparison = (
    degrees.join(departures, "id", "left")
    .join(ranks, "id")
    .join(vertices.select("id", "airport_name", "city", "state"), "id")
    .select(
        "id",
        "airport_name",
        "city",
        "state",
        "degree",
        F.coalesce(F.col("departing_flights"), F.lit(0)).alias("departing_flights"),
        F.round("pagerank", 4).alias("pagerank"),
    )
    .orderBy(F.desc("pagerank"))
)
print("Degree, departing flights, and PageRank")
display(comparison.limit(15))


## 14. Connected components and communities

`connectedComponents` groups airports that can reach each other when direction is ignored. `labelPropagation` assigns a community label. On a dense route graph the result is often one large community plus a few small ones. Labels can change between runs.

Both algorithms use the checkpoint directory created in the setup cell.


In [ ]:
components = graph.connectedComponents().select("id", "component")
component_sizes = components.groupBy("component").count().orderBy(F.desc("count"))
print("Connected components")
display(component_sizes)

communities = graph.labelPropagation(maxIter=5).select("id", "label")
community_sizes = communities.groupBy("label").count().orderBy(F.desc("count"))
print(f"Communities: {community_sizes.count():,}")
display(community_sizes.limit(10))


## 15. Degree and PageRank bars

The bars use the IATA code. Degree is on the left and PageRank is on the right. The two rankings can differ because PageRank follows who points at an airport, and degree only counts routes.


In [ ]:
import matplotlib.pyplot as plt

degree_top = (
    degrees.join(vertices, "id")
    .select("id", "degree")
    .orderBy(F.desc("degree"))
    .limit(20)
    .toPandas()
)
pagerank_top = (
    ranks.join(vertices, "id")
    .select("id", "pagerank")
    .orderBy(F.desc("pagerank"))
    .limit(20)
    .toPandas()
)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].bar(range(len(degree_top)), degree_top["degree"], color="steelblue")
axes[0].set_xticks(range(len(degree_top)))
axes[0].set_xticklabels(degree_top["id"], rotation=45, ha="right")
axes[0].set_title("Top 20 airports by degree")
axes[0].set_ylabel("Degree")

axes[1].bar(range(len(pagerank_top)), pagerank_top["pagerank"], color="steelblue")
axes[1].set_xticks(range(len(pagerank_top)))
axes[1].set_xticklabels(pagerank_top["id"], rotation=45, ha="right")
axes[1].set_title("Top 20 airports by PageRank")
axes[1].set_ylabel("PageRank")

fig.tight_layout()
plt.show()


## 16. Hubs by region

These lists use degree inside three groups of states. An airport with a null state is one of the flight airports missing from the airports table, so it does not appear here.


In [ ]:
print("Airports in the flight graph, by state")
display(vertices.groupBy("state").count().orderBy(F.desc("count")).limit(10))

northeast = ["NY", "NJ", "PA", "MA", "CT", "RI", "VT", "NH", "ME"]
southwest = ["TX", "AZ", "NM", "NV"]
west_coast = ["CA", "WA", "OR"]

ranked = degrees.join(vertices, "id").select("id", "airport_name", "city", "state", "degree")

print("Northeast")
display(ranked.filter(F.col("state").isin(northeast)).orderBy(F.desc("degree")).limit(5))

print("Southwest")
display(ranked.filter(F.col("state").isin(southwest)).orderBy(F.desc("degree")).limit(5))

print("West coast")
display(ranked.filter(F.col("state").isin(west_coast)).orderBy(F.desc("degree")).limit(5))


## 17. Summary

The totals below are taken from the cached edge frame. Leave this Spark session running if you want to keep querying `graph`, `vertices`, or `edges`.


In [ ]:
totals = edges.agg(
    F.sum("flight_count").alias("total_flights"),
    F.avg("flight_count").alias("avg_flights_per_route"),
).collect()[0]

print(f"Airports in the flight graph: {vertices.count():,}")
print(f"Routes: {edges.count():,}")
print(f"Flights: {totals['total_flights']:,.0f}")
print(f"Average flights per route: {totals['avg_flights_per_route']:.2f}")
print(f"Airports with coordinates: {vertices.filter(F.col('latitude').isNotNull()).count():,}")
print("Spark session is still running.")
